In [1]:
import os

In [2]:
%pwd

'd:\\codes\\mlflow\\data-science-project\\reserch'

In [3]:
os.chdir("../")
%pwd

'd:\\codes\\mlflow\\data-science-project'

In [24]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelTrainerConfig:
    root_dir: Path
    train_data_path: Path
    test_data_path: Path
    model_name: str
    alpha: float
    l1_ratio: float
    target_column: str


In [28]:
from src.datascienceproject.constants import *
from src.datascienceproject.utils.common import read_yaml, create_directories

class ConfiguationManager:
    def __init__(self, 
                 config_pathfile=CONFIG_FILE_PATH,
                 params_pathfile=PARAMS_FILE_PATH,
                 schema_pathfile=SCHEMA_FILE_PATH):
        self.config=read_yaml(config_pathfile)
        self.params=read_yaml(params_pathfile)
        self.schema=read_yaml(schema_pathfile)
        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self)->ModelTrainerConfig:
        config=self.config.model_trainer
        params=self.params.ElasticNet
        schema=self.schema.TARGET_COLUMN
        create_directories([config.root_dir])

        model_trainer_config=ModelTrainerConfig(root_dir=config.root_dir,
                                                train_data_path=config.train_data_path,
                                                test_data_path=config.test_data_path,
                                                model_name=config.model_name,
                                                alpha=params.alpha,
                                                l1_ratio=params.l1_ratio,
                                                target_column=schema.name)
        return model_trainer_config

In [29]:
import os
from src.datascienceproject import logger
from sklearn.linear_model import ElasticNet
import joblib
import pandas as pd

class ModelTrainer:
    def __init__(self,config:ModelTrainerConfig):
            self.config=config

    def train(self):
        train_data=pd.read_csv(self.config.train_data_path)
        test_data=pd.read_csv(self.config.test_data_path)
        train_x =train_data.drop([self.config.target_column], axis=1)
        test_x =test_data.drop([self.config.target_column], axis=1)
        train_y =train_data[[self.config.target_column]]
        test_y =test_data[[self.config.target_column]]
        
        lr = ElasticNet(alpha=self.config.alpha,l1_ratio= self.config.l1_ratio, random_state=42)
        lr.fit(train_x, train_y)

        joblib.dump(lr, os.path.join(self.config.root_dir, self.config.model_name))

In [30]:
try:
    config=ConfiguationManager()
    model_trainer_config=config.get_model_trainer_config()
    model_trainer = ModelTrainer(config=model_trainer_config)
    model_trainer.train()
except Exception as e:
    raise e

[2026-09-14 14:55:06,162: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-09-14 14:55:06,166: INFO: common: yaml file: params.yaml loaded successfully]
[2026-09-14 14:55:06,171: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-09-14 14:55:06,173: INFO: common: created directory at: artifacts]
[2026-09-14 14:55:06,178: INFO: common: created directory at: artifacts/model_trainer]
